<a href="https://colab.research.google.com/github/LUNALEE002/-/blob/main/%E1%84%89%E1%85%B5%E1%86%AF%E1%84%92%E1%85%A5%E1%86%B7_AI%E1%84%83%E1%85%AE%E1%84%80%E1%85%A2%E1%84%92%E1%85%A1%E1%86%B8%E1%84%8E%E1%85%B5%E1%84%80%E1%85%B5_model_merging_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 AI 두 개 합치기 — Model Merging 실험

**가중치 수술(weight surgery)의 가장 쉽고 안전한 첫 실험.**
두 AI 모델의 *가중치를 수학적으로 섞어서* 두 능력을 다 가진 모델 하나를 만든다.
학습(GPU 오래 돌리기) 없이, 몇 분이면 끝난다.

> 💡 원리: AI의 능력은 가중치 공간의 '방향(벡터)'이다. 두 모델의 가중치를 보간(SLERP)하면 두 능력이 한 모델에 섞인다.

**준비**: 위 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 (무료)

## 1단계 — 도구 설치 (mergekit)

In [ ]:
!pip install -q torch transformers accelerate
!git clone -q https://github.com/arcee-ai/mergekit.git
%cd mergekit
!pip install -q -e .
print('✅ mergekit 설치 완료')

/content/mergekit
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 45.7 MB/s eta 0:00:00
  Building editable for mergekit (pyproject.toml) ..

## 2단계 — 합칠 두 모델 + 방법 정하기

⚠️ **두 모델은 반드시 같은 베이스(같은 아키텍처·크기)여야** 합쳐진다.
아래는 작은 Qwen2.5-1.5B 계열 예시 (무료 T4에서 안전). 원하는 같은-베이스 모델로 바꿔도 된다.

- `MODEL_A`, `MODEL_B`: 합칠 두 모델 (HuggingFace 이름)
- `merge_method: slerp`: 두 가중치를 매끄럽게 보간 (가장 무난한 방법)
- `t: 0.5`: 0.5 = 반반. 0에 가까우면 A 쪽, 1에 가까우면 B 쪽.

In [ ]:
config = '''
slices:
  - sources:
      - model: Qwen/Qwen2.5-1.5B-Instruct
        layer_range: [0, 28]
      - model: Qwen/Qwen2.5-Coder-1.5B-Instruct
        layer_range: [0, 28]
merge_method: slerp
base_model: Qwen/Qwen2.5-1.5B-Instruct
parameters:
  t: 0.5
dtype: bfloat16
'''
with open('merge_config.yaml', 'w') as f:
    f.write(config)
print('✅ 설정 저장 — 일반 Qwen(대화) + Coder Qwen(코딩)을 반반 섞는다')

✅ 설정 저장 — 일반 Qwen(대화) + Coder Qwen(코딩)을 반반 섞는다


## 3단계 — 합치기 실행 (수술!) 🔪

두 모델을 받아서 가중치를 섞는다. 처음엔 모델 다운로드 때문에 몇 분 걸린다.

In [ ]:
!mergekit-yaml merge_config.yaml ./merged_model --cuda --allow-crimes
print('✅ 합치기 완료 → ./merged_model 에 새 모델 탄생')

config.json: 100% 660/660 [00:00<00:00, 4.10MB/s]
config.json: 100% 660/660 [00:00<00:00, 4.67MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Warmup loader cache:   0% 0/2 [00:00<?, ?it/s]

Fetching 6 files:   0% 0/6 [00:00<?, ?it/s]

Fetching 6 files:  33% 2/6 [00:00<00:00,  7.85it/s]

Fetching 6 files: 100% 6/6 [00:10<00:00,  1.78s/it]

Warmup loader cache:  50% 1/2 [00:11<00:11, 11.15s/it]



Fetching 6 files:   0% 0/6 [00:00<?, ?it/s]







Fetching 6 files:  33% 2/6 [00:00<00:00,  7.17it/s]












Fetching 6 files: 100% 6/6 [00:09<00:00,  1.56s/it]


Warmup loader cache: 100% 2/2 [00:21<00:00, 10.53s/it]
Download complete: 100% 3.10G/3.10G [00:21<00:00, 147MB/s]
Download complete: 100% 3.10G/3.10G [00:09<00:00, 313MB/s]
Executing graph: 100% 1692/1692 [00:32<00:00, 52.07it/s]
Fetching 6 files: 100% 6/6 [00:00<00:00, 40524.68it/s]
Download complete: : 0.00B [00:00, ?B/s]              

Fetching 6 files: 100% 6/6 [00:00<00:00, 36105.92it/s]

Download com

## 4단계 — 합쳐진 모델 테스트 🧪

대화도 되고 코딩도 되는지 — 두 능력이 한 모델에 들어왔는지 확인.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tok = AutoTokenizer.from_pretrained('./merged_model')
model = AutoModelForCausalLM.from_pretrained('./merged_model', torch_dtype=torch.bfloat16, device_map='auto')

def ask(q):
    msgs = [{'role':'user','content':q}]
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    print(tok.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))

print('=== 대화 능력 ==='); ask('1인 기업가에게 좋은 아침 루틴 3가지 추천해줘')
print('\n=== 코딩 능력 ==='); ask('파이썬으로 피보나치 수열 함수 짧게 써줘')

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== 대화 능력 ===
일상적인 정상화와 건강한体液을 위해, 3가지의 아침 routines를 3가드릴정의 1-20min/회/주요/3가지/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정상/3가정

=== 코딩 능력 ===
当然可以理解你的需求。这里提供一个简单的斐波那契数列生成函数的示例：

```python
def fibonacci(n):
    if n <= 2:
        return [0, 1]
    else:
        a, b = 0, 1
        for i in range(3, n + 1):
            a, b = b, (a + b) % 1000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000


## 5단계 (선택) — 우리 앱(Connect AI)에서 돌리기 🖥️

합친 모델을 GGUF로 변환하면 Connect AI 데스크톱 앱에서 바로 실행할 수 있다.
(llama.cpp 변환 → 양자화 → 앱의 🤖 내 AI 에서 불러오기)

In [ ]:
%cd /content
!git clone -q https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
# FP16 GGUF로 변환
!python llama.cpp/convert_hf_to_gguf.py /content/mergekit/merged_model --outfile my-merged.gguf --outtype f16
print('✅ my-merged.gguf 생성 — 다운로드해서 Connect AI 앱 모델 폴더에 넣으면 끝')

/content
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 82.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 150.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 136.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 7.0 MB/s eta 0:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## 🎓 무슨 일이 일어난 건가

- 두 모델의 **가중치를 SLERP(구면 선형 보간)로 섞었다** — 학습이 아니라 수학적 합성
- 결과: "대화 잘하는 Qwen" + "코딩 잘하는 Qwen" = **둘 다 하는 모델**
- 이게 **Model Soups / TIES / DARE** 계열 연구(weight 산술)의 기초

**다음 실험 아이디어:**
- `t` 값을 0.3, 0.7로 바꿔서 어느 능력이 세지는지 비교
- 한국어 모델 + 코딩 모델 합치기
- `merge_method`를 `ties`나 `dare_ties`로 바꿔보기 (3개 이상도 합쳐짐)

> 🔬 이건 '검열 제거(abliteration)'와 다른, 안전하고 건설적인 weight 수술이다. 능력을 **더하는** 방향.